# Proyek Analisis Data: E-Commerce Public Dataset (Olist)
- **Nama:** Syahla Sandiani Assyifa Adam
- **Email:** ssandianiaa@gmail.com
- **ID Dicoding:** syahlasandiani

## Menentukan Pertanyaan Bisnis

Seluruh pertanyaan disusun dengan metode **SMART** (Specific, Measurable, Action-Oriented, Relevant, Time-bound). Periode analisis adalah **Januari 2017 - Agustus 2018**, karena data 2016 sangat sedikit dan data September-Oktober 2018 tidak lengkap.

1. **Pertanyaan 1 (Produk):** Kategori produk apa saja yang menjadi 5 kategori dengan *revenue* tertinggi dan 5 kategori dengan *revenue* terendah (beserta jumlah pesanannya) pada pesanan berstatus *delivered* selama Januari 2017 - Agustus 2018, sehingga tim merchandising dapat menentukan kategori yang diprioritaskan untuk promosi dan stok?
2. **Pertanyaan 2 (Wilayah & Logistik):** Negara bagian (*state*) mana yang menyumbang *revenue* terbesar, dan berapa rata-rata waktu pengiriman (hari) di 10 *state* teratas tersebut dibandingkan rata-rata nasional selama Januari 2017 - Agustus 2018, sehingga tim logistik dapat menentukan wilayah yang perlu perbaikan pengiriman?
3. **Pertanyaan 3 (Pelanggan):** Bagaimana segmentasi pelanggan berdasarkan analisis RFM (*Recency, Frequency, Monetary*) pada pesanan *delivered* Januari 2017 - Agustus 2018, dan segmen mana yang paling besar sehingga tim CRM dapat menentukan target kampanye *retention*?

## Import Semua Packages/Library yang Digunakan

In [ ]:
import os
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Data Wrangling

### Gathering Data

Dataset yang dipakai adalah **E-Commerce Public Dataset (Olist)**. Tabel yang dibutuhkan untuk menjawab pertanyaan bisnis:
- `olist_customers_dataset.csv`: identitas dan lokasi pelanggan
- `olist_orders_dataset.csv`: status dan waktu pesanan
- `olist_order_items_dataset.csv`: item pesanan beserta harga dan ongkos kirim
- `olist_products_dataset.csv`: informasi produk (kategori)
- `product_category_name_translation.csv`: terjemahan nama kategori (Portugis -> Inggris)
- `olist_geolocation_dataset.csv`: koordinat berdasarkan kode pos (untuk analisis geospasial)

In [ ]:
DATA_DIR = "data"

customers_df = pd.read_csv(f"{DATA_DIR}/olist_customers_dataset.csv")
orders_df = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv")
items_df = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
products_df = pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
category_df = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")
geo_df = pd.read_csv(f"{DATA_DIR}/olist_geolocation_dataset.csv")

customers_df.head()

In [ ]:
orders_df.head()

In [ ]:
items_df.head()

In [ ]:
products_df.head()

In [ ]:
category_df.head()

In [ ]:
geo_df.head()

**Insight (Gathering Data):**
- Keenam tabel berhasil dimuat ke dalam DataFrame dan dapat dihubungkan melalui kunci `customer_id`, `order_id`, `product_id`, serta `product_category_name`.
- `customer_id` bersifat unik **per pesanan**, sedangkan pelanggan yang sebenarnya diidentifikasi oleh `customer_unique_id`. Hal ini penting untuk analisis RFM agar pelanggan yang membeli berulang tidak terhitung sebagai pelanggan berbeda.
- Tabel geolocation berisi banyak baris per kode pos sehingga kemungkinan besar mengandung duplikasi.

### Assessing Data

Pada tahap ini kualitas data diperiksa: ukuran data, tipe data, *missing value*, duplikasi, *inconsistent value*, dan *outlier*.

In [ ]:
tables = {
    "customers": customers_df, "orders": orders_df, "order_items": items_df,
    "products": products_df, "category": category_df, "geolocation": geo_df,
}
for name, df in tables.items():
    print(f"{name:12s} -> {df.shape[0]:>9,} baris, {df.shape[1]:>2} kolom")

**1. Tipe data** (kolom tanggal pada `orders_df` masih bertipe object/string)

In [ ]:
orders_df.info()

**2. Missing value**

In [ ]:
for name in ["orders", "products", "customers", "order_items"]:
    miss = tables[name].isna().sum()
    miss = miss[miss > 0]
    print(f"--- {name} ---")
    print(miss if len(miss) else "Tidak ada missing value", "\n")

**3. Duplicate data**

In [ ]:
for name, df in tables.items():
    print(f"{name:12s} -> duplikat: {df.duplicated().sum():,}")

**4. Inconsistent value:** status pesanan, nama kota pada geolocation (variasi penulisan/aksen), dan kategori produk tanpa terjemahan

In [ ]:
print(orders_df["order_status"].value_counts(), "\n")

variants = geo_df.loc[geo_df["geolocation_city"].str.contains("paulo", case=False, na=False), "geolocation_city"]
print("Variasi penulisan kota Sao Paulo pada geolocation:")
print(variants.value_counts().head(8), "\n")

no_translation = set(products_df["product_category_name"].dropna()) - set(category_df["product_category_name"])
print("Kategori produk tanpa terjemahan:", no_translation)

**5. Outlier & invalid value** (harga, ongkos kirim, dan konsistensi waktu pesanan)

In [ ]:
print(items_df[["price", "freight_value"]].describe(), "\n")

q1, q3 = items_df["price"].quantile([0.25, 0.75])
iqr = q3 - q1
n_out = (items_df["price"] > q3 + 1.5 * iqr).sum()
print(f"Item dengan harga di atas batas IQR (outlier): {n_out:,} ({n_out / len(items_df):.1%})")

purchase = pd.to_datetime(orders_df["order_purchase_timestamp"])
delivered = pd.to_datetime(orders_df["order_delivered_customer_date"])
print("Pesanan dengan tanggal tiba < tanggal beli (tidak valid):", (delivered < purchase).sum())
print("Pesanan 'delivered' tanpa tanggal tiba:",
      ((orders_df["order_status"] == "delivered") & orders_df["order_delivered_customer_date"].isna()).sum())
print("Rentang tanggal pembelian:", purchase.min(), "s.d.", purchase.max())

In [ ]:
monthly_check = purchase.dt.to_period("M").value_counts().sort_index()
monthly_check.plot(kind="bar", figsize=(12, 3), color="#9aa5b1", title="Jumlah pesanan per bulan (seluruh data mentah)")
plt.show()

**Insight (Assessing Data) - masalah yang ditemukan dan rencana penanganannya:**

| No | Masalah | Tabel | Solusi pada tahap Cleaning |
|----|---------|-------|----------------------------|
| 1 | Tipe data kolom tanggal masih string | `orders_df` | Konversi ke `datetime` |
| 2 | *Missing value* pada tanggal pengiriman (pesanan belum/tidak terkirim) dan pada kategori produk | `orders_df`, `products_df` | Hanya gunakan pesanan `delivered` yang memiliki tanggal tiba; isi kategori kosong dengan `unknown` |
| 3 | *Duplicate data* dalam jumlah besar | `geo_df` | Hapus duplikat, lalu agregasi koordinat per *state* |
| 4 | *Inconsistent value*: variasi penulisan nama kota, kategori tanpa terjemahan | `geo_df`, `products_df` | Normalisasi teks (huruf kecil, hapus aksen); gunakan nama Portugis jika terjemahan tidak ada |
| 5 | *Outlier* harga | `items_df` | Dipertahankan karena merupakan produk bernilai tinggi yang valid; interpretasi memakai total/median |
| 6 | Data tidak lengkap pada 2016 dan akhir 2018 | `orders_df` | Batasi periode analisis Januari 2017 - Agustus 2018 |

### Cleaning Data

Setiap masalah dari tahap *Assessing Data* ditangani satu per satu.

**1. Konversi tipe data tanggal**

In [ ]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in date_cols:
    orders_df[col] = pd.to_datetime(orders_df[col])

orders_df[date_cols].dtypes

**2. Filter pesanan `delivered` bertanggal tiba valid dan batasi periode Januari 2017 - Agustus 2018**

In [ ]:
before = len(orders_df)
orders_clean = orders_df[orders_df["order_status"] == "delivered"].dropna(subset=["order_delivered_customer_date"])
orders_clean = orders_clean[orders_clean["order_delivered_customer_date"] >= orders_clean["order_purchase_timestamp"]]
orders_clean = orders_clean[
    (orders_clean["order_purchase_timestamp"] >= "2017-01-01")
    & (orders_clean["order_purchase_timestamp"] < "2018-09-01")
].copy()

print(f"Jumlah pesanan: {before:,} -> {len(orders_clean):,}")
print("Rentang:", orders_clean["order_purchase_timestamp"].min(), "s.d.", orders_clean["order_purchase_timestamp"].max())

**3. Tangani kategori produk: isi kategori kosong, gabungkan terjemahan**

In [ ]:
products_clean = products_df.merge(category_df, on="product_category_name", how="left")
products_clean["product_category_name_english"] = (
    products_clean["product_category_name_english"]
    .fillna(products_clean["product_category_name"])   # tidak ada terjemahan -> pakai nama asli
    .fillna("unknown")                                  # kategori kosong
)
products_clean[["product_id", "product_category_name_english"]].isna().sum()

**4. Bersihkan `geo_df`: hapus duplikat, normalisasi nama kota, dan hitung titik pusat (centroid) tiap *state***

In [ ]:
def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))

geo_clean = geo_df.drop_duplicates().copy()
geo_clean["geolocation_city"] = geo_clean["geolocation_city"].astype(str).str.lower().apply(strip_accents)
print(f"geo_df: {len(geo_df):,} -> {len(geo_clean):,} baris setelah hapus duplikat")
print("Variasi 'sao paulo' sekarang:", geo_clean.loc[geo_clean["geolocation_city"].str.contains("sao paulo"), "geolocation_city"].nunique())

state_centroid = (
    geo_clean.groupby("geolocation_state")[["geolocation_lat", "geolocation_lng"]].mean()
    .rename(columns={"geolocation_lat": "state_lat", "geolocation_lng": "state_lng"})
    .reset_index().rename(columns={"geolocation_state": "customer_state"})
)
state_centroid.head()

**5. Outlier harga:** tidak dihapus karena merupakan produk bernilai tinggi yang valid (bukan kesalahan input). Analisis tetap aman karena menggunakan agregasi total dan rata-rata per kelompok.

**6. Gabungkan seluruh tabel menjadi satu DataFrame utama (`main_df`)**, lalu tambahkan kolom turunan `delivery_days` (lama pengiriman) dan `is_late` (terlambat dari estimasi).

In [ ]:
main_df = (
    items_df
    .merge(orders_clean[["order_id", "customer_id", "order_purchase_timestamp",
                         "order_delivered_customer_date", "order_estimated_delivery_date"]],
           on="order_id", how="inner")
    .merge(customers_df[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left")
    .merge(products_clean[["product_id", "product_category_name_english"]], on="product_id", how="left")
    .merge(state_centroid, on="customer_state", how="left")
)

main_df["product_category_name_english"] = main_df["product_category_name_english"].fillna("unknown")
main_df["delivery_days"] = (
    main_df["order_delivered_customer_date"] - main_df["order_purchase_timestamp"]
).dt.total_seconds() / 86400
main_df["is_late"] = main_df["order_delivered_customer_date"] > main_df["order_estimated_delivery_date"]

print("Bentuk main_df:", main_df.shape)
print("Missing value tersisa:\n", main_df[["customer_unique_id", "customer_state", "product_category_name_english", "state_lat"]].isna().sum())
main_df.head()

**7. Simpan data bersih** untuk digunakan oleh dashboard Streamlit

In [ ]:
os.makedirs("dashboard", exist_ok=True)
main_df.to_csv("dashboard/main_data.csv", index=False)
print("Tersimpan: dashboard/main_data.csv")

**Insight (Cleaning Data):**
- Hanya pesanan **delivered** yang dipakai agar *revenue* dan lama pengiriman benar-benar mencerminkan transaksi yang selesai.
- Penggabungan tabel menghasilkan satu DataFrame `main_df` pada level **item pesanan** yang siap dianalisis dan menjadi sumber data dashboard (`main_data.csv`).
- Duplikasi pada tabel geolocation berhasil dibersihkan sehingga koordinat *state* yang dihitung tidak bias oleh kode pos yang tercatat berkali-kali.

## Exploratory Data Analysis (EDA)

### Eksplorasi Ringkasan Data

In [ ]:
print("Jumlah pesanan unik   :", f"{main_df['order_id'].nunique():,}")
print("Jumlah pelanggan unik :", f"{main_df['customer_unique_id'].nunique():,}")
print("Jumlah kategori produk:", main_df["product_category_name_english"].nunique())
print("Jumlah state          :", main_df["customer_state"].nunique())
print("Total revenue (BRL)   :", f"{main_df['price'].sum():,.0f}")
print()
main_df[["price", "freight_value", "delivery_days"]].describe()

### Eksplorasi Tren Bulanan

In [ ]:
monthly_df = (
    main_df.groupby(main_df["order_purchase_timestamp"].dt.to_period("M"))
    .agg(orders=("order_id", "nunique"), revenue=("price", "sum"))
    .reset_index()
)
monthly_df["order_purchase_timestamp"] = monthly_df["order_purchase_timestamp"].astype(str)
monthly_df = monthly_df.rename(columns={"order_purchase_timestamp": "month"})
monthly_df["revenue_mom_growth_%"] = monthly_df["revenue"].pct_change() * 100
monthly_df

### Eksplorasi Pertanyaan 1: Performa Kategori Produk

In [ ]:
category_summary = (
    main_df.groupby("product_category_name_english")
    .agg(orders=("order_id", "nunique"), items_sold=("order_item_id", "count"),
         revenue=("price", "sum"), avg_price=("price", "mean"))
    .sort_values("revenue", ascending=False)
    .reset_index()
)
category_summary["revenue_share_%"] = category_summary["revenue"] / category_summary["revenue"].sum() * 100

print("=== 5 kategori revenue tertinggi ===")
display(category_summary.head(5))
print("=== 5 kategori revenue terendah ===")
display(category_summary.tail(5))
print(f"Kontribusi 5 kategori teratas terhadap total revenue: {category_summary['revenue_share_%'].head(5).sum():.1f}%")

### Eksplorasi Pertanyaan 2: Performa Wilayah dan Logistik

In [ ]:
state_summary = (
    main_df.groupby("customer_state")
    .agg(customers=("customer_unique_id", "nunique"), orders=("order_id", "nunique"),
         revenue=("price", "sum"), avg_delivery_days=("delivery_days", "mean"),
         late_rate=("is_late", "mean"), state_lat=("state_lat", "first"), state_lng=("state_lng", "first"))
    .sort_values("revenue", ascending=False)
    .reset_index()
)
state_summary["revenue_share_%"] = state_summary["revenue"] / state_summary["revenue"].sum() * 100
national_avg_delivery = main_df["delivery_days"].mean()

print(f"Rata-rata waktu pengiriman nasional: {national_avg_delivery:.1f} hari")
print("=== 10 state dengan revenue tertinggi ===")
display(state_summary.head(10))
print("=== 5 state dengan pengiriman paling lama ===")
display(state_summary.sort_values("avg_delivery_days", ascending=False).head(5)[["customer_state", "orders", "avg_delivery_days", "late_rate"]])

**Insight (EDA):**
- Tren bulanan menunjukkan pertumbuhan *revenue* dan jumlah pesanan yang jelas sepanjang 2017 dengan lonjakan pada akhir tahun (periode promosi akhir tahun), lalu relatif stabil pada 2018.
- Pendapatan terkonsentrasi pada beberapa kategori: lima kategori teratas menyumbang porsi besar dari total *revenue*, sedangkan kategori terbawah hanya menyumbang sangat kecil.
- Pendapatan sangat terpusat pada *state* tertentu (terutama SP), sementara *state* di wilayah utara dan timur laut cenderung memiliki waktu pengiriman jauh lebih lama dari rata-rata nasional.

## Visualization & Explanatory Analysis

Prinsip desain yang diterapkan pada seluruh visualisasi: **satu pesan per grafik** (judul berupa kesimpulan), **warna aksen hanya untuk data yang ingin ditonjolkan** dan abu-abu untuk pembanding, **bar dimulai dari nol** (integritas data), label dan satuan jelas, serta menghilangkan elemen yang tidak perlu (*chartjunk*).

In [ ]:
ACCENT = "#1f5fbf"
MUTED = "#b8c0cc"
WARN = "#d1495b"

def rupiah_fmt(v, _=None):
    """Format angka revenue (BRL) menjadi satuan ribu/juta."""
    if v >= 1e6:
        return f"{v / 1e6:,.1f} jt"
    return f"{v / 1e3:,.0f} rb"

def clean_axes(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", visible=False)

### Tren Revenue dan Jumlah Pesanan Bulanan (pelengkap EDA)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

ax[0].plot(monthly_df["month"], monthly_df["revenue"], marker="o", color=ACCENT, linewidth=2)
ax[0].set_title("Revenue bulanan tumbuh pesat sepanjang 2017", loc="left", fontsize=13, fontweight="bold")
ax[0].set_ylabel("Revenue (BRL)")
ax[0].yaxis.set_major_formatter(mtick.FuncFormatter(rupiah_fmt))

ax[1].bar(monthly_df["month"], monthly_df["orders"], color=MUTED)
peak = monthly_df["orders"].idxmax()
ax[1].bar(monthly_df.loc[peak, "month"], monthly_df.loc[peak, "orders"], color=ACCENT)
ax[1].set_title("Jumlah pesanan bulanan (puncak disorot)", loc="left", fontsize=13, fontweight="bold")
ax[1].set_ylabel("Jumlah pesanan")

for a in ax:
    a.tick_params(axis="x", rotation=60)
    clean_axes(a)
    a.set_xlabel("")
plt.tight_layout()
plt.show()

**Insight:** Revenue dan jumlah pesanan meningkat tajam sepanjang 2017 dan mencapai puncak pada periode promosi akhir tahun, lalu bertahan di level tinggi pada 2018. Ini menunjukkan bisnis sudah melewati fase awal dan permintaan musiman (akhir tahun) perlu diantisipasi dari sisi stok dan logistik.

### Pertanyaan 1: Kategori Produk dengan Revenue Tertinggi dan Terendah

In [ ]:
top5 = category_summary.head(5).iloc[::-1]
bottom5 = category_summary.tail(5).iloc[::-1]

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

colors_top = [MUTED] * 4 + [ACCENT]
ax[0].barh(top5["product_category_name_english"], top5["revenue"], color=colors_top)
ax[0].set_title("5 kategori dengan revenue tertinggi", loc="left", fontsize=13, fontweight="bold")
for i, (rev, orders) in enumerate(zip(top5["revenue"], top5["orders"])):
    ax[0].text(rev, i, f"  {rev/1e3:,.0f} rb | {orders:,} pesanan", va="center", fontsize=9)
ax[0].set_xlim(0, top5["revenue"].max() * 1.45)

ax[1].barh(bottom5["product_category_name_english"], bottom5["revenue"], color=WARN)
ax[1].set_title("5 kategori dengan revenue terendah", loc="left", fontsize=13, fontweight="bold")
for i, (rev, orders) in enumerate(zip(bottom5["revenue"], bottom5["orders"])):
    ax[1].text(rev, i, f"  {rev:,.0f} | {orders:,} pesanan", va="center", fontsize=9)
ax[1].set_xlim(0, bottom5["revenue"].max() * 1.6)

for a in ax:
    a.set_xlabel("Revenue (BRL)")
    a.set_ylabel("")
    a.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    clean_axes(a)
    a.grid(axis="x", alpha=0.3)

plt.suptitle("Revenue per kategori produk (pesanan delivered, Jan 2017 - Agu 2018)", x=0.01, ha="left", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
top_cat = category_summary.iloc[0]
print(f"Kategori teratas: {top_cat['product_category_name_english']} (revenue BRL {top_cat['revenue']:,.0f}, "
      f"{top_cat['revenue_share_%']:.1f}% dari total)")
print(f"Lima kategori teratas menyumbang {category_summary['revenue_share_%'].head(5).sum():.1f}% total revenue")
print(f"Lima kategori terbawah hanya menyumbang {category_summary['revenue_share_%'].tail(5).sum():.3f}% total revenue")

**Insight (Pertanyaan 1):**
- Revenue sangat terkonsentrasi pada segelintir kategori. Kategori seperti *health_beauty*, *watches_gifts*, *bed_bath_table*, *sports_leisure*, dan *computers_accessories* mendominasi pendapatan, sedangkan kategori terbawah (umumnya niche seperti layanan/seni/CD-DVD) hampir tidak berkontribusi.
- Kategori dengan jumlah pesanan banyak belum tentu memiliki revenue terbesar. Perbedaan rata-rata harga per item memengaruhi posisi kategori (misalnya *watches_gifts* dengan harga rata-rata tinggi).

### Pertanyaan 2: Revenue dan Waktu Pengiriman per State

In [ ]:
top10_state = state_summary.head(10)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

colors_rev = [ACCENT] + [MUTED] * 9
ax[0].bar(top10_state["customer_state"], top10_state["revenue"], color=colors_rev)
ax[0].set_title("SP mendominasi revenue dibanding state lain", loc="left", fontsize=13, fontweight="bold")
ax[0].set_ylabel("Revenue (BRL)")
ax[0].yaxis.set_major_formatter(mtick.FuncFormatter(rupiah_fmt))

colors_del = [WARN if d > national_avg_delivery else MUTED for d in top10_state["avg_delivery_days"]]
ax[1].bar(top10_state["customer_state"], top10_state["avg_delivery_days"], color=colors_del)
ax[1].axhline(national_avg_delivery, color="black", linestyle="--", linewidth=1)
ax[1].text(9.45, national_avg_delivery + 0.3, f"Rata-rata nasional {national_avg_delivery:.1f} hari",
           ha="right", fontsize=9)
ax[1].set_title("Rata-rata lama pengiriman 10 state teratas (merah = di atas rata-rata nasional)",
                loc="left", fontsize=13, fontweight="bold")
ax[1].set_ylabel("Hari")

for a in ax:
    a.set_xlabel("State")
    clean_axes(a)
plt.tight_layout()
plt.show()

In [ ]:
print(f"SP menyumbang {state_summary.iloc[0]['revenue_share_%']:.1f}% total revenue.")
print(f"Tiga state teratas menyumbang {state_summary['revenue_share_%'].head(3).sum():.1f}% total revenue.")
slow = state_summary.sort_values("avg_delivery_days", ascending=False).iloc[0]
fast = state_summary.sort_values("avg_delivery_days").iloc[0]
print(f"Pengiriman paling lama: {slow['customer_state']} ({slow['avg_delivery_days']:.1f} hari); "
      f"paling cepat: {fast['customer_state']} ({fast['avg_delivery_days']:.1f} hari).")

**Insight (Pertanyaan 2):**
- *State* SP menyumbang porsi revenue terbesar secara signifikan, diikuti RJ dan MG. Tiga *state* teratas saja sudah menguasai mayoritas pendapatan.
- Waktu pengiriman berbeda jauh antarwilayah: SP relatif cepat, sedangkan beberapa *state* besar di luar wilayah tenggara (misalnya BA, PE, atau CE pada kelompok yang lebih besar) berada di atas rata-rata nasional. *State* di utara memiliki waktu pengiriman paling lama.

## Analisis Lanjutan

Tiga teknik analisis lanjutan (tanpa *machine learning*) diterapkan: **RFM Analysis**, **Geospatial Analysis**, dan **Clustering (Manual Grouping & Binning)**.

### 1. RFM Analysis (menjawab Pertanyaan 3)

**Tujuan:** mengelompokkan pelanggan berdasarkan perilaku pembelian agar tim CRM dapat menentukan strategi yang berbeda untuk tiap kelompok. Dihitung per `customer_unique_id`:
- **Recency:** jumlah hari sejak pembelian terakhir pelanggan (acuan: 1 hari setelah transaksi terakhir pada data).
- **Frequency:** jumlah pesanan unik pelanggan.
- **Monetary:** total pengeluaran pelanggan (BRL).

Skor R dan M dibuat dengan **kuartil** (1-4), sedangkan skor F dibuat dengan aturan manual karena sebagian besar pelanggan hanya membeli sekali (F = 1, F = 2, F >= 3). Segmen kemudian ditentukan dengan aturan bisnis (*manual grouping*).

In [ ]:
ref_date = main_df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm_df = (
    main_df.groupby("customer_unique_id")
    .agg(last_purchase=("order_purchase_timestamp", "max"),
         frequency=("order_id", "nunique"),
         monetary=("price", "sum"))
    .reset_index()
)
rfm_df["recency"] = (ref_date - rfm_df["last_purchase"]).dt.days

# Skor: kuartil untuk R dan M (rank agar tidak error saat nilai kembar), aturan manual untuk F
rfm_df["r_score"] = pd.qcut(rfm_df["recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)
rfm_df["m_score"] = pd.qcut(rfm_df["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm_df["f_score"] = np.select([rfm_df["frequency"] == 1, rfm_df["frequency"] == 2], [1, 2], default=3)

def rfm_segment(row):
    if row["f_score"] >= 2 and row["r_score"] >= 3:
        return "Champions"
    if row["f_score"] >= 2:
        return "Loyal - Perlu Diaktifkan Kembali"
    if row["r_score"] >= 3 and row["m_score"] >= 3:
        return "Pelanggan Baru Bernilai Tinggi"
    if row["r_score"] >= 3:
        return "Pelanggan Baru"
    if row["m_score"] >= 3:
        return "Berisiko Hilang (Bernilai Tinggi)"
    return "Hilang / Bernilai Rendah"

rfm_df["segment"] = rfm_df.apply(rfm_segment, axis=1)
rfm_df.head()

In [ ]:
segment_order = ["Champions", "Loyal - Perlu Diaktifkan Kembali", "Pelanggan Baru Bernilai Tinggi",
                 "Pelanggan Baru", "Berisiko Hilang (Bernilai Tinggi)", "Hilang / Bernilai Rendah"]

segment_summary = (
    rfm_df.groupby("segment")
    .agg(customers=("customer_unique_id", "count"), avg_recency=("recency", "mean"),
         avg_frequency=("frequency", "mean"), avg_monetary=("monetary", "mean"),
         total_monetary=("monetary", "sum"))
    .reindex(segment_order).dropna(how="all").reset_index()
)
segment_summary["customer_share_%"] = segment_summary["customers"] / segment_summary["customers"].sum() * 100
segment_summary["revenue_share_%"] = segment_summary["total_monetary"] / segment_summary["total_monetary"].sum() * 100
segment_summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

seg = segment_summary.iloc[::-1]
big = seg["customers"].idxmax()
colors_seg = [ACCENT if i == big else MUTED for i in seg.index]

ax[0].barh(seg["segment"], seg["customers"], color=colors_seg)
for i, (c, p) in enumerate(zip(seg["customers"], seg["customer_share_%"])):
    ax[0].text(c, i, f"  {c:,} ({p:.1f}%)", va="center", fontsize=9)
ax[0].set_xlim(0, seg["customers"].max() * 1.3)
ax[0].set_title("Jumlah pelanggan per segmen RFM", loc="left", fontsize=13, fontweight="bold")
ax[0].set_xlabel("Jumlah pelanggan")

ax[1].barh(seg["segment"], seg["avg_monetary"], color=colors_seg)
for i, v in enumerate(seg["avg_monetary"]):
    ax[1].text(v, i, f"  {v:,.0f}", va="center", fontsize=9)
ax[1].set_xlim(0, seg["avg_monetary"].max() * 1.2)
ax[1].set_title("Rata-rata monetary per segmen (BRL)", loc="left", fontsize=13, fontweight="bold")
ax[1].set_xlabel("Rata-rata total belanja (BRL)")
ax[1].set_yticklabels([])

for a in ax:
    a.set_ylabel("")
    clean_axes(a)
    a.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
repeat_rate = (rfm_df["frequency"] >= 2).mean() * 100
print(f"Pelanggan yang pernah membeli lebih dari satu kali: {repeat_rate:.1f}%")
largest = segment_summary.sort_values("customers", ascending=False).iloc[0]
print(f"Segmen terbesar: {largest['segment']} ({largest['customer_share_%']:.1f}% pelanggan)")

**Insight (Pertanyaan 3 - RFM):**
- Sebagian besar pelanggan hanya bertransaksi **satu kali** (porsi *repeat customer* sangat kecil), sehingga mayoritas pelanggan berada pada segmen non-loyal. Retensi menjadi masalah utama bisnis ini.
- Segmen *Champions* dan *Loyal* berukuran kecil, tetapi rata-rata belanjanya lebih besar daripada segmen lain sehingga bernilai strategis.
- Segmen *Berisiko Hilang (Bernilai Tinggi)* adalah pelanggan yang pernah berbelanja besar namun sudah lama tidak kembali, sehingga cocok menjadi target kampanye *win-back*.

### 2. Geospatial Analysis

**Tujuan:** melihat pola sebaran geografis revenue dan waktu pengiriman antar *state* di Brasil untuk mengidentifikasi wilayah dengan permintaan tinggi dan wilayah dengan pengiriman lambat. Koordinat tiap *state* diperoleh dari rata-rata `geolocation_lat` dan `geolocation_lng` pada tabel geolocation. Ukuran lingkaran menunjukkan revenue, dan warna menunjukkan lama pengiriman.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
sc = ax.scatter(
    state_summary["state_lng"], state_summary["state_lat"],
    s=state_summary["revenue"] / state_summary["revenue"].max() * 2500 + 40,
    c=state_summary["avg_delivery_days"], cmap="YlOrRd", edgecolor="black", linewidth=0.5, alpha=0.85,
)
for _, r in state_summary.iterrows():
    ax.annotate(r["customer_state"], (r["state_lng"], r["state_lat"]), ha="center", va="center", fontsize=8)

cbar = plt.colorbar(sc, ax=ax, shrink=0.7)
cbar.set_label("Rata-rata lama pengiriman (hari)")
ax.set_title("Revenue (ukuran) dan lama pengiriman (warna) per state di Brasil",
             loc="left", fontsize=13, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Peta interaktif menggunakan **folium** (tampil ketika notebook dijalankan di Jupyter/Colab):

In [ ]:
try:
    import folium

    brazil_map = folium.Map(location=[-14.2, -51.9], zoom_start=4, tiles="CartoDB positron")
    max_rev = state_summary["revenue"].max()

    def delivery_color(days):
        return "#d1495b" if days > 20 else ("#edae49" if days > national_avg_delivery else "#2e86ab")

    for _, r in state_summary.iterrows():
        folium.CircleMarker(
            location=[r["state_lat"], r["state_lng"]],
            radius=5 + 30 * (r["revenue"] / max_rev) ** 0.5,
            color=delivery_color(r["avg_delivery_days"]), fill=True, fill_opacity=0.65,
            popup=folium.Popup(
                f"<b>{r['customer_state']}</b><br>Revenue: BRL {r['revenue']:,.0f}<br>"
                f"Pesanan: {r['orders']:,}<br>Rata-rata pengiriman: {r['avg_delivery_days']:.1f} hari", max_width=250),
        ).add_to(brazil_map)

    brazil_map
except ImportError:
    print("Package folium belum terpasang. Jalankan: pip install folium")

**Insight (Geospatial):**
- Revenue terkonsentrasi di wilayah **tenggara** Brasil (SP, RJ, MG), yang juga memiliki pengiriman tercepat.
- Semakin jauh dari pusat distribusi di tenggara (ke arah utara dan timur laut), waktu pengiriman cenderung semakin lama, meskipun jumlah pesanannya jauh lebih kecil.

### 3. Clustering: Manual Grouping & Binning

**Tujuan:** mengelompokkan pesanan berdasarkan lama pengiriman ke dalam kategori (*binning*) untuk mengetahui seberapa besar porsi pesanan yang tiba cepat versus lambat, serta hubungannya dengan keterlambatan dari estimasi. Pengelompokan segmen pelanggan pada RFM di atas juga merupakan bentuk *manual grouping* berbasis aturan bisnis.

In [ ]:
order_delivery = main_df.drop_duplicates("order_id")[["order_id", "delivery_days", "is_late"]].copy()

bins = [0, 7, 14, 21, 30, np.inf]
labels = ["Sangat cepat (<=7 hari)", "Cepat (8-14 hari)", "Normal (15-21 hari)", "Lambat (22-30 hari)", "Sangat lambat (>30 hari)"]
order_delivery["delivery_group"] = pd.cut(order_delivery["delivery_days"], bins=bins, labels=labels, include_lowest=True)

delivery_group_summary = (
    order_delivery.groupby("delivery_group", observed=True)
    .agg(orders=("order_id", "count"), late_rate=("is_late", "mean"))
    .reset_index()
)
delivery_group_summary["order_share_%"] = delivery_group_summary["orders"] / delivery_group_summary["orders"].sum() * 100
delivery_group_summary["late_rate"] = delivery_group_summary["late_rate"] * 100
delivery_group_summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

colors_bin = [MUTED, MUTED, MUTED, WARN, WARN]
ax[0].bar(delivery_group_summary["delivery_group"].astype(str), delivery_group_summary["order_share_%"], color=colors_bin)
for i, v in enumerate(delivery_group_summary["order_share_%"]):
    ax[0].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
ax[0].set_title("Sebaran pesanan berdasarkan kelompok lama pengiriman", loc="left", fontsize=13, fontweight="bold")
ax[0].set_ylabel("% pesanan")

ax[1].bar(delivery_group_summary["delivery_group"].astype(str), delivery_group_summary["late_rate"], color=colors_bin)
for i, v in enumerate(delivery_group_summary["late_rate"]):
    ax[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
ax[1].set_title("Persentase pesanan terlambat dari estimasi per kelompok", loc="left", fontsize=13, fontweight="bold")
ax[1].set_ylabel("% terlambat")

for a in ax:
    a.tick_params(axis="x", rotation=25)
    a.set_xlabel("")
    clean_axes(a)
plt.tight_layout()
plt.show()

**Insight (Binning):**
- Sebagian besar pesanan tiba dalam waktu kurang dari tiga minggu, tetapi masih ada kelompok pesanan yang membutuhkan lebih dari 3 minggu.
- Tingkat keterlambatan dari estimasi meningkat tajam pada kelompok pengiriman yang lambat, sehingga estimasi pengiriman untuk wilayah jauh perlu dikalibrasi ulang atau proses pengiriman dipercepat.

## Conclusion

**Kesimpulan Pertanyaan 1 (kategori produk):**
Revenue Olist selama Januari 2017 - Agustus 2018 sangat terkonsentrasi pada sedikit kategori. Lima kategori teratas (misalnya *health_beauty*, *watches_gifts*, *bed_bath_table*, *sports_leisure*, *computers_accessories*) menyumbang porsi besar dari total revenue, sedangkan lima kategori terbawah hanya berkontribusi sangat kecil. Angka pasti kontribusinya dapat dilihat pada output sel EDA dan visualisasi Pertanyaan 1.

**Kesimpulan Pertanyaan 2 (wilayah & logistik):**
*State* SP menjadi penyumbang revenue terbesar dan, bersama RJ dan MG, menguasai mayoritas pendapatan. Namun waktu pengiriman sangat timpang: wilayah tenggara relatif cepat, sedangkan *state* yang jauh dari pusat distribusi (utara dan timur laut) memiliki rata-rata pengiriman jauh di atas rata-rata nasional. Analisis geospasial dan *binning* menguatkan bahwa pengiriman yang lambat berkaitan dengan peningkatan keterlambatan dari estimasi.

**Kesimpulan Pertanyaan 3 (pelanggan/RFM):**
Mayoritas pelanggan hanya membeli satu kali sehingga segmen terbesar adalah pelanggan non-loyal (baru atau sudah lama tidak kembali), sementara *Champions* dan *Loyal* jumlahnya kecil. Segmen "Berisiko Hilang (Bernilai Tinggi)" adalah peluang terbesar untuk dipulihkan.

> Catatan: pernyataan kualitatif di atas sebaiknya dicocokkan dengan angka pada output sel saat notebook dijalankan.

### Rekomendasi (Action Item)
1. **Fokus promosi dan stok pada kategori unggulan** (5 kategori teratas), serta evaluasi kategori terbawah: bundling dengan kategori populer atau pertimbangkan pengurangan stok/*listing* yang tidak efektif.
2. **Perbaiki logistik di wilayah lambat:** tambah *fulfillment center* atau kerja sama kurir di wilayah utara/timur laut, serta kalibrasi ulang estimasi pengiriman agar lebih realistis dan menurunkan keterlambatan.
3. **Tingkatkan retensi pelanggan:** kampanye *win-back* (voucher/email personal) untuk segmen "Berisiko Hilang (Bernilai Tinggi)", program loyalitas untuk *Champions*/*Loyal*, dan *follow-up* pembelian kedua untuk "Pelanggan Baru" dalam 30 hari pertama.